In [ ]:
#  -- Noah Russell -- 2/7/26
# Text preprocessing, TF-IDF, and cosine similarity using NLTK
# Designed to run in Google Colab without downloading extra NLTK corpora

import os, math
import numpy as np
import pandas as pd
import nltk
from nltk.tokenize import RegexpTokenizer
from nltk.stem import PorterStemmer
from nltk.probability import FreqDist

# ----------------------------
# 1) Load documents
# These are the three news articles provided by the professor
# ----------------------------
DOC_PATHS = {
    "100554newsML": "/content/100554newsML.txt",
    "100593newsML": "/content/100593newsML.txt",
    "100618newsML": "/content/100618newsML.txt",
}

# Quick check so if a file name is slightly off, we get a helpful message
missing = [p for p in DOC_PATHS.values() if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(
        "Could not find these file paths in Colab:\n"
        + "\n".join(missing)
        + "\n\nTip: Make sure the .txt files are uploaded into /content (Colab file sidebar)."
    )

docs_raw = {}
for name, path in DOC_PATHS.items():
    with open(path, "r", encoding="utf-8") as f:
        docs_raw[name] = f.read()

print("Loaded documents:")
for name, text in docs_raw.items():
    print(f"- {name}: {len(text)} characters")

# ----------------------------
# 2) Text preprocessing
# Tokenization, stopword removal, and stemming using NLTK
# RegexpTokenizer avoids needing the 'punkt' download
# Stopwords are defined manually to keep everything offline
# ----------------------------
tokenizer = RegexpTokenizer(r"[A-Za-z]+")
stemmer = PorterStemmer()

# Small built-in stopword set so we don't need nltk.download("stopwords")
STOP_WORDS = set("""
i me my myself we our ours ourselves you youre youve youll youd your yours yourself yourselves
he him his himself she shes her hers herself it its itself they them their theirs themselves
what which who whom this that these those am is are was were be been being have has had having
do does did doing a an the and but if or because as until while of at by for with about
against between into through during before after above below to from up down in out on off
over under again further then once here there when where why how all any both each few more
most other some such no nor not only own same so than too very s t can will just don should now
d ll m o re ve y ain aren couldnt didn doesnt hadn hasnt haven isnt ma mightnt mustnt neednt
shan shouldnt wasnt werent won wouldnt
""".split())

def preprocess(text: str):
    # tokenize text and convert to lowercase
    tokens = tokenizer.tokenize(text.lower())

    # remove common stop words
    tokens_nostop = [w for w in tokens if w not in STOP_WORDS]

    # apply Porter stemming
    stems = [stemmer.stem(w) for w in tokens_nostop]

    return tokens, tokens_nostop, stems

docs_tokens = {}
docs_nostop = {}
docs_stems = {}

for name, text in docs_raw.items():
    t, ns, s = preprocess(text)
    docs_tokens[name] = t
    docs_nostop[name] = ns
    docs_stems[name] = s

# Screenshot-friendly samples
for name in DOC_PATHS.keys():
    print(f"\n=== {name} samples ===")
    print("Tokens (first 25):", docs_tokens[name][:25])
    print("After stopword removal (first 25):", docs_nostop[name][:25])
    print("After stemming (first 25):", docs_stems[name][:25])

# ----------------------------
# 3) TF-IDF calculation and document-word matrix
# TF = term count / total terms in document
# IDF = log((N + 1) / (df + 1)) + 1  (smoothed)
# NLTK's FreqDist is used to compute term frequencies
# ----------------------------
doc_names = list(DOC_PATHS.keys())
N = len(doc_names)

# Term counts per document
freqs = {name: FreqDist(docs_stems[name]) for name in doc_names}

# Vocabulary across all docs
vocab = sorted(set(stem for name in doc_names for stem in docs_stems[name]))

# Document frequency (df): how many docs contain each term
df = {term: sum(1 for name in doc_names if term in freqs[name]) for term in vocab}

# Smoothed IDF
idf = {term: math.log((N + 1) / (df[term] + 1)) + 1 for term in vocab}

# Build TF-IDF matrix
tfidf = np.zeros((N, len(vocab)), dtype=float)

for i, name in enumerate(doc_names):
    fdist = freqs[name]
    total_terms = sum(fdist.values())

    for j, term in enumerate(vocab):
        if term in fdist:
            tf = fdist[term] / total_terms
            tfidf[i, j] = tf * idf[term]

tfidf_df = pd.DataFrame(tfidf, index=doc_names, columns=vocab)

print("\nTF-IDF matrix shape (docs x terms):", tfidf_df.shape)

# Show top terms per doc (nice for screenshots)
for name in doc_names:
    top_terms = tfidf_df.loc[name].sort_values(ascending=False).head(15)
    print(f"\nTop TF-IDF terms for {name}:")
    print(top_terms)

# Save doc-word matrix to CSV
tfidf_df.to_csv("tfidf_doc_word_matrix.csv", index=True)
print("\nSaved full TF-IDF document-word matrix -> tfidf_doc_word_matrix.csv")

# ----------------------------
# 4) Pairwise cosine similarity
# Cosine similarity compares documents by direction rather than size,
# which makes it a good fit for TF-IDF vectors
# ----------------------------
def cosine_sim(vec_a, vec_b):
    num = float(np.dot(vec_a, vec_b))
    den = float(np.linalg.norm(vec_a) * np.linalg.norm(vec_b))
    return 0.0 if den == 0 else num / den

sim = np.zeros((N, N), dtype=float)
for i in range(N):
    for j in range(N):
        sim[i, j] = cosine_sim(tfidf[i], tfidf[j])

sim_df = pd.DataFrame(sim, index=doc_names, columns=doc_names)

print("\n=== Pairwise Cosine Similarity (final result) ===")
print(sim_df.round(6))

# Save similarity matrix to CSV
sim_df.to_csv("cosine_similarity_matrix.csv", index=True)
print("\nSaved cosine similarity matrix -> cosine_similarity_matrix.csv")


Loaded documents:
- 100554newsML: 4025 characters
- 100593newsML: 2808 characters
- 100618newsML: 2776 characters

=== 100554newsML samples ===
Tokens (first 25): ['channel', 'tunnel', 'operator', 'eurotunnel', 'on', 'monday', 'announced', 'details', 'of', 'a', 'deal', 'giving', 'bank', 'creditors', 'percent', 'of', 'the', 'company', 'in', 'return', 'for', 'wiping', 'out', 'billion', 'pounds']
After stopword removal (first 25): ['channel', 'tunnel', 'operator', 'eurotunnel', 'monday', 'announced', 'details', 'deal', 'giving', 'bank', 'creditors', 'percent', 'company', 'return', 'wiping', 'billion', 'pounds', 'billion', 'massive', 'debts', 'long', 'awaited', 'highly', 'complex', 'restructuring']
After stemming (first 25): ['channel', 'tunnel', 'oper', 'eurotunnel', 'monday', 'announc', 'detail', 'deal', 'give', 'bank', 'creditor', 'percent', 'compani', 'return', 'wipe', 'billion', 'pound', 'billion', 'massiv', 'debt', 'long', 'await', 'highli', 'complex', 'restructur']

=== 100593newsML